In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
tg = pd.read_csv('API_Date_Final.csv')
site = pd.read_csv('Parsing_final (2).csv')

In [ ]:
tg.head(5)

In [ ]:
site.head(5)

In [ ]:
tg.info()

In [ ]:
site.info()

In [ ]:
topics = {
    'Политика': ['трамп', 'президент','путин','министр','санкц','выборы','правительств'],
    'Война': ['украин', 'атак','удар','обстрел','войск','иран','израил','ракет'],
    'Экономика': ['рубл', 'доллар','инфляц','ставк','бюджет','банк','ввп'],
    'Бизнес': ['компани', 'рынк','акци','миллиард','бизнес','сделк'],
    'Технологии': ['цифров', 'технолог','интернет','платформ','искусственн']
}

In [ ]:
def get_topic(text):
    if pd.isna(text):
        return 'Другое'
    t = text.lower()
    for topic, kws in topic.items():
        if any(k in t for k in kws):
            return topic
    return 'Другое'

In [ ]:
tg['topic'] = tg['content'].apply(get_topic)
site['topic'] = site['title'].apply(get_topic)

In [ ]:
print('Темы в Тг:')
tg['topic'].value_counts()

In [ ]:
print('Темы на сайте:')
site['topic'].value_counts()

In [ ]:
Частота тем на сайте

In [ ]:
site_topic = site['topic'].value_counts()

plt.figure(figsize=(16, 5))
plt.bar(site_topic.index,site_topic.values)
plt.xlabel('Тема')
plt.ylabel('Кол-во статей')

In [ ]:
Количество публикаций на сайте по частям

In [ ]:
site['dt'] = pd.to_datetime(site['date'])
site['hour'] = site['dt'].dt.hour

site_hours = site.groupby('hour').size()

plt.figure(figsize=(16, 5))
plt.bar(site_hours.index, site_hours.values)
plt.xlabel('Час')
plt.ylabel('Кол-во статей')
plt.xticks(range(0, 24))
plt.show()

In [ ]:
Количество статей по теме и часу публикации

In [ ]:
site_pivot = site.groupby(['hour', 'topic']).size().unstack(fill_value=0)

plt.figure(figsize=(16, 5))
sns.heatmap(site_pivot.T, cmap='Blues', annot=True, fmt='.0f')
plt.xlabel('Час')
plt.ylabel('Тема')
plt.show()

In [ ]:
Средние просмотры по теме

In [ ]:
tg_topic_views = tg.groupby('topic')['views'].mean().sort_values(ascending=False).round(0)

plt.figure(figsize=(16, 5))
plt.bar(tg_topic_views.index, tg_topic_views.values)
plt.xlabel('Тема')
plt.ylabel('Среднее число просмотров')
for i, j in enumerate(tg_topic_views.values):
    plt.text(i, j, j)
plt.show()

In [ ]:
Средние реакции по теме

In [ ]:
tg_topic_react = tg.groupby('topic')['reactions_count'].mean().sort_values(ascending=False).round(2)

plt.figure(figsize=(16, 5))
plt.bar(tg_topic_react.index, tg_topic_react.values)
plt.xlabel('Тема')
plt.ylabel('Среднее число реакций')
for i, j in enumerate(tg_topic_react.values):
    plt.text(i, j, j)
plt.show()

In [ ]:
Просмотры по часам

In [ ]:
tg.['dt'] = pd.to_datetime(tg['date'])
tg['hour'] = tg['dt'].dt.hour

tg_hour_views = tg.groupby('hour')['views'].mean().round(0)

plt.figure(figsize=(16, 5))
plt.bar(tg_hour_views.index, tg_hour_views.values)
plt.xlabel('Час суток')
plt.ylabel('Среднее число просмотров')
plt.xticks(sorted(tg_hour_views.index))
plt.show()

In [ ]:
Реакции по часу публикации

In [ ]:
tg_hour_react = tg.groupby('hour')['reactions_count'].mean().round(1)

plt.figure(figsize=(16, 5))
plt.bar(tg_hour_react.index, tg_hour_react.values)
plt.xlabel('Час')
plt.ylabel('Среднее число реакций')
plt.xticks(sorted(tg_hour_react.index))
plt.show()

In [ ]:
Просмотры по теме и часу

In [ ]:
tg_pivot = tg.groupby(['hour', 'topic'])['views'].mean().round(0).unstack(fill_value=0)

plt.figure(figsize=(20, 5))
sns.heatmap(tg_pivot.T, cmap='Blues', annot=True, fmt='.0f')
plt.xlabel('Час')
plt.ylabel('Тема')
plt.show()

In [ ]:
Число просмотров в зависимости от длины текста

In [ ]:
tg['text_len'] = tg['content'].str.len()
bins_text  = [0, 300, 500, 700, 1000, 5000]
labels_text = ['< 300', '300–500', '500–700', '700–1000', '> 1000']
tg['text_bin'] = pd.cut(tg['text_len'], bins=bins_text, labels=labels_text)

In [ ]:
text_views = tg.groupby('text_bin', observed=True)['views'].mean().round(0)

plt.figure(figsize=(10, 5))
plt.bar(text_views.index, text_views.values)
plt.xlabel('Длина текста')
plt.ylabel('Среднее число просмотров')
plt.show()

In [ ]:
Корреляция показателей телеграмма

In [ ]:
corr_cols = ['views', 'reactions_count', 'forwards', 'text_len', 'has_media']
corr_labels = ['Просмотры', 'Реакции', 'Репосты', 'Длина текста', 'Есть медиа']

corr_matrix = tg[corr_cols].corr()
corr_matrix.index = corr_labels
corr_matrix.columns = corr_labels

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', vmin=-1, vmax=1)
plt.show()